In [8]:
import os
from dotenv import load_dotenv

load_dotenv() 

API_KEY = os.getenv("FRED_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "FRED_API_KEY not found. Create a .env file or set an environment variable."
    )

In [9]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key=API_KEY)

START = '1986-01-01'
END   = '2026-01-01'

# --- Macro series (monthly) ---
series_monthly = {
    'CPI':            'CPIAUCSL',
    'TB3MS':          'TB3MS',
    'M1':             'M1SL', # changed definition in 2020
    'M2':             'M2SL',
}

# --- Exchange rates (daily → resample to monthly) ---
series_fx = {
    'AUD_USD': 'DEXUSAL',
    'CAD_USD': 'DEXCAUS',
    'NZD_USD': 'DEXUSNZ',
    'ZAR_USD': 'DEXSFUS', # only starts 1994
}

# Pull monthly macro
macro_df = pd.DataFrame({
    name: fred.get_series(sid, observation_start=START, observation_end=END)
    for name, sid in series_monthly.items()
})

# Pull FX and resample to monthly mean
fx_df = pd.DataFrame({
    name: fred.get_series(sid, observation_start=START, observation_end=END)
           .resample('MS').mean()          # Month-start, mean of daily values
    for name, sid in series_fx.items()
})

# Combine
df = macro_df.join(fx_df, how='outer')
df.index = pd.to_datetime(df.index)
df = df.resample('MS').last()             # Align all to month-start frequency

print(df.head())

              CPI  TB3MS     M1      M2   AUD_USD   CAD_USD   NZD_USD  \
1986-01-01  109.9   7.07  621.4  2502.1  0.700005  1.406981  0.516571   
1986-02-01  109.7   7.06  625.2  2512.9  0.699284  1.404284  0.531774   
1986-03-01  109.1   6.56  633.5  2533.1  0.707938  1.400910  0.528200   
1986-04-01  108.7   6.06  641.0  2557.8  0.722841  1.387868  0.561268   
1986-05-01  109.0   6.15  652.0  2584.8  0.727195  1.375719  0.566657   

             ZAR_USD  
1986-01-01  2.362762  
1986-02-01  2.089716  
1986-03-01  2.040943  
1986-04-01  2.051609  
1986-05-01  2.194029  


In [ ]:
df.index.name = 'Date'
df.head()

CPI        float64
TB3MS      float64
M1         float64
M2         float64
AUD_USD    float64
CAD_USD    float64
NZD_USD    float64
ZAR_USD    float64
dtype: object

In [7]:
df.to_csv("predictor_data.csv")